# Product Embedding — Thu nghiem offline
Thu nghiem tao vector embedding cho san pham bep truoc khi dua vao `app/models/product_embedding.py`.

- **Khong co torch**: TF-IDF fallback
- **Co torch**: Vietnamese-SBERT / PhoBERT mean-pooling (dim=768)

In [ ]:
import numpy as np
import pickle, os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

try:
    import torch
    from transformers import AutoTokenizer, AutoModel
    TORCH_AVAILABLE = True
    print("torch", torch.__version__, "available")
except ImportError:
    TORCH_AVAILABLE = False
    print("torch not available — TF-IDF fallback")


## 1. Du lieu mau — san pham bep

In [ ]:
PRODUCTS = [
    {"id":"p1","name":"Noi com dien Panasonic 1.8L",
     "desc":"Noi com 1.8 lit, nau hoi, giu am tu dong","cat":"Thiet bi nau nuong"},
    {"id":"p2","name":"Noi com dien Toshiba 1.0L",
     "desc":"Noi com mini 1 lit, tiet kiem dien, 1-2 nguoi","cat":"Thiet bi nau nuong"},
    {"id":"p3","name":"Chao chong dinh Sunhouse 28cm",
     "desc":"Chao chien ceramic 28cm, dung duoc bep tu","cat":"Chao va noi"},
    {"id":"p4","name":"Chao gang Lodge 26cm",
     "desc":"Chao gang duc nguyen khoi, giu nhiet tot","cat":"Chao va noi"},
    {"id":"p5","name":"Dao bep Nhat Kai 18cm",
     "desc":"Dao thai thit khong gi cao cap, can go","cat":"Dao va dung cu"},
    {"id":"p6","name":"Bo dao nha bep 5 mon",
     "desc":"Set dao thai got chat keo va thot khong gi","cat":"Dao va dung cu"},
    {"id":"p7","name":"May xay sinh to Philips 750W",
     "desc":"May xay 750W 1.5L 3 toc do luoi titan","cat":"May xay va ep"},
    {"id":"p8","name":"May ep cham Hurom H-AA",
     "desc":"May ep cold-press 43RPM giu vitamin enzyme","cat":"May xay va ep"},
]

texts = [p["name"]+" "+p["desc"]+" "+p["cat"] for p in PRODUCTS]
ids   = [p["id"] for p in PRODUCTS]
print(len(texts), "san pham ready")


## 2. TF-IDF Embedding (fallback — luon chay duoc)

In [ ]:
vec = TfidfVectorizer(ngram_range=(1,2), min_df=1)
tfidf_mat = vec.fit_transform(texts).toarray()
tfidf_emb = {pid: tfidf_mat[i] for i, pid in enumerate(ids)}
print("TF-IDF shape:", tfidf_mat.shape, "vocab:", len(vec.vocabulary_))


## 3. Vietnamese-SBERT Embedding (neu co torch)

In [ ]:
bert_emb = {}
if TORCH_AVAILABLE:
    MODEL = "keepitreal/vietnamese-sbert"
    print("Loading", MODEL, "...")
    tok = AutoTokenizer.from_pretrained(MODEL)
    mdl = AutoModel.from_pretrained(MODEL)
    mdl.eval()

    def mean_pool(out, mask):
        t = out.last_hidden_state
        m = mask.unsqueeze(-1).expand(t.size()).float()
        return (t * m).sum(1) / m.sum(1).clamp(min=1e-9)

    for pid, txt in zip(ids, texts):
        enc = tok(txt, return_tensors="pt", truncation=True,
                  max_length=128, padding=True)
        with torch.no_grad():
            o = mdl(**enc)
        e = mean_pool(o, enc["attention_mask"]).squeeze().numpy()
        bert_emb[pid] = e
    print("SBERT OK, dim:", e.shape[0])
else:
    print("Bo qua — dung TF-IDF")

active = bert_emb if TORCH_AVAILABLE else tfidf_emb
emb_mat = np.array([active[pid] for pid in ids])
print("Matrix:", emb_mat.shape)


## 4. Test cosine similarity — san pham tuong tu

In [ ]:
sim = cosine_similarity(emb_mat)

def show_similar(qid, k=3):
    i = ids.index(qid)
    ranked = np.argsort(sim[i])[::-1]
    pname = PRODUCTS[i]["name"]
    print("\nTuong tu", pname + ":")
    for ri in ranked[1:k+1]:
        print("  %.3f  %s" % (sim[i][ri], PRODUCTS[ri]["name"]))

show_similar("p1")  # noi com -> noi com
show_similar("p5")  # dao -> bo dao
show_similar("p7")  # may xay -> may ep


## 5. Visualize PCA 2D

In [ ]:
pca = PCA(n_components=2)
coords = pca.fit_transform(emb_mat)
cats = list({p["cat"] for p in PRODUCTS})
cmap = {c: plt.cm.tab10(i/len(cats)) for i,c in enumerate(cats)}

fig, ax = plt.subplots(figsize=(10,7))
for i, p in enumerate(PRODUCTS):
    ax.scatter(*coords[i], color=cmap[p["cat"]], s=120, zorder=3)
    ax.annotate(p["name"][:22], coords[i], fontsize=8, ha="left", va="bottom")
model_lbl = "SBERT" if TORCH_AVAILABLE else "TF-IDF"
ax.set_title("Product Embeddings (%s) — PCA 2D | var=%.1f%%" % (
    model_lbl, pca.explained_variance_ratio_.sum()*100))
plt.tight_layout(); plt.show()


## 6. Export pkl — production
File duoc load boi `RecommendationEngine` tu `settings.EMBEDDINGS_PATH/product_embeddings.pkl`

In [ ]:
OUT = "../app/data/embeddings"
os.makedirs(OUT, exist_ok=True)
path = os.path.join(OUT, "product_embeddings_sample.pkl")
with open(path, "wb") as f: pickle.dump(active, f)
with open(path, "rb") as f: loaded = pickle.load(f)
print("Saved", len(loaded), "embeddings ->", path)
print("dim:", list(loaded.values())[0].shape)
